# Tournament Simulation

In this notebook, we use the trained Poisson models to simulate the full 2026 World Cup.

The goal is to:

1. Load the trained home-goals and away-goals models
2. Build match-level predictions for any two teams
3. Simulate group-stage matches
4. Rank teams inside each group
5. Simulate the knockout bracket
6. Run the tournament many times using Monte Carlo simulation
7. Estimate each team's probability of reaching each stage

In [1]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path

# Section 1: Load models and data

In [2]:
# Project paths
DATA_DIR = Path("../data")
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
REFERENCE_DIR = DATA_DIR / "reference"
MODELS_DIR = Path("../models")

In [3]:
# Load trained Poisson models
with open(MODELS_DIR / "poisson_home.pkl", "rb") as f:
    model_home = pickle.load(f)

with open(MODELS_DIR / "poisson_away.pkl", "rb") as f:
    model_away = pickle.load(f)

# Load feature column order used during training
with open(MODELS_DIR / "feature_columns.pkl", "rb") as f:
    feature_columns = pickle.load(f)

print("Models loaded successfully.")
print(f"Number of model features: {len(feature_columns)}")

Models loaded successfully.
Number of model features: 17


In [4]:
# Core datasets
df_historical = pd.read_csv(INTERIM_DIR / "historical_matches.csv")
df_fixtures = pd.read_csv(INTERIM_DIR / "wc2026_fixtures.csv")

# Processed feature datasets
df_match_features = pd.read_csv(PROCESSED_DIR / "df_match_features.csv")
df_form = pd.read_csv(PROCESSED_DIR / "df_form_2026.csv")
df_h2h = pd.read_csv(PROCESSED_DIR / "df_h2h_2026.csv")

# Reference datasets
df_confederations = pd.read_csv(REFERENCE_DIR / "FIFA_confederations.csv")
df_knockout = pd.read_csv(REFERENCE_DIR / "fixtures_knockout_wc2026.csv")

# group_stages.csv uses semicolon separator
df_groups = pd.read_csv(REFERENCE_DIR / "group_stages.csv", sep=";")

# FIFA ranking reference
df_fifa_rank = pd.read_csv(DATA_DIR / "raw" / "wc_2026_48_teams_fifa_rank_change_corrected.csv")

print("Datasets loaded successfully.")

Datasets loaded successfully.


In [ ]:
# Inspect shape
datasets = {
    "historical_matches": df_historical,
    "wc2026_fixtures": df_fixtures,
    "df_match_features": df_match_features,
    "df_form_2026": df_form,
    "df_h2h_2026": df_h2h,
    "FIFA_confederations": df_confederations,
    "group_stages": df_groups,
    "fixtures_knockout_wc2026": df_knockout,
    "fifa_rank_2026": df_fifa_rank,
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

historical_matches: (49215, 9)
wc2026_fixtures: (72, 9)
df_match_features: (49048, 16)
df_form_2026: (48, 2)
df_h2h_2026: (565, 4)
FIFA_confederations: (48, 2)
group_stages: (48, 3)
fixtures_knockout_wc2026: (32, 8)
fifa_rank_2026: (48, 4)


In [7]:
# Inspect columns
for name, df in datasets.items():
    print(f"\n{name}")
    print(df.columns.tolist())


historical_matches
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']

wc2026_fixtures
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']

df_match_features
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'home_elo_pre', 'away_elo_pre', 'elo_diff', 'tournament_weight', 'is_competitive', 'home_confederation', 'away_confederation']

df_form_2026
['team', 'form_score']

df_h2h_2026
['team_a', 'team_b', 'h2h_score', 'n_meetings_analysed']

FIFA_confederations
['nation', 'confederation']

group_stages
['group', 'position', 'nation']

fixtures_knockout_wc2026
['match_id', 'round', 'match_date', 'match_time', 'home_slot', 'away_slot', 'winner_advances_to', 'loser_advances_to']

fifa_rank_2026
['Nation', 'FIFA_2022_ranking', 'FIFA_2026_rank', 'rank_change']


In [8]:
# Check number of World Cup teams
teams_from_groups = set(df_groups["nation"])
teams_from_confederations = set(df_confederations["nation"])
teams_from_form = set(df_form["team"])
teams_from_fifa = set(df_fifa_rank["Nation"])

print("Teams in group_stages:", len(teams_from_groups))
print("Teams in confederations:", len(teams_from_confederations))
print("Teams in form table:", len(teams_from_form))
print("Teams in FIFA ranking table:", len(teams_from_fifa))

# Check missing teams across key reference tables
print("\nTeams in groups but missing from confederations:")
print(sorted(teams_from_groups - teams_from_confederations))

print("\nTeams in groups but missing from form table:")
print(sorted(teams_from_groups - teams_from_form))

print("\nTeams in groups but missing from FIFA ranking table:")
print(sorted(teams_from_groups - teams_from_fifa))

Teams in group_stages: 48
Teams in confederations: 48
Teams in form table: 48
Teams in FIFA ranking table: 48

Teams in groups but missing from confederations:
[]

Teams in groups but missing from form table:
[]

Teams in groups but missing from FIFA ranking table:
[]


In [9]:
display(df_fixtures.head())
display(df_groups.head())
display(df_knockout.head())
display(df_match_features.head())

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,2026-06-11,Mexico,South Africa,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False
1,2026-06-11,South Korea,Czech Republic,NaN,NaN,FIFA World Cup,Zapopan,Mexico,True
2,2026-06-12,Canada,Bosnia and Herzegovina,NaN,NaN,FIFA World Cup,Toronto,Canada,False
3,2026-06-12,United States,Paraguay,NaN,NaN,FIFA World Cup,Inglewood,United States,False
4,2026-06-13,Qatar,Switzerland,NaN,NaN,FIFA World Cup,Santa Clara,United States,True


,group,position,nation
0,A,1,Mexico
1,A,2,South Africa
2,A,3,South Korea
3,A,4,Czech Republic
4,B,1,Canada


,match_id,round,match_date,match_time,home_slot,away_slot,winner_advances_to,loser_advances_to
0,M73,R32,2026-06-28,20:00,2A,2B,M90,NaN
1,M74,R32,2026-06-29,21:30,1E,3ABCDF,M89,NaN
2,M75,R32,2026-06-30,02:00,1F,2C,M90,NaN
3,M76,R32,2026-06-29,18:00,1C,2F,M91,NaN
4,M77,R32,2026-06-30,22:00,1I,3CDFGH,M89,NaN


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo_pre,away_elo_pre,elo_diff,tournament_weight,is_competitive,home_confederation,away_confederation
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1500.000000,1500.000000,0.000000,1,False,UEFA,UEFA
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,1502.801300,1497.198700,5.602600,1,False,UEFA,UEFA
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1486.622531,1513.377469,-26.754938,1,False,UEFA,UEFA
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,1505.454946,1494.545054,10.909891,1,False,UEFA,UEFA
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1497.633108,1502.366892,-4.733783,1,False,UEFA,UEFA


#
----

# Section 2: create lookup tables for Elo, form, H2H, and confederation.

The aim here is to make it easy to fetch, for any team:
- Elo rating
- confederation
- recent form score
- FIFA rank
- H2H score against another team

## Section 2 — Create lookup tables

Before we can predict any 2026 match, we need quick lookup tables for team-level and matchup-level information.

In this section, we create:

- latest Elo rating per team
- team-to-confederation mapping
- team-to-form-score mapping
- team-to-FIFA-rank mapping
- head-to-head lookup between two teams

These lookup tables will make the prediction and simulation functions much cleaner.

In [11]:
#### prepare latest Elo per team

# Make sure dates are treated as dates
df_match_features["date"] = pd.to_datetime(df_match_features["date"])

# Home-team Elo records
home_elo = df_match_features[["date", "home_team", "home_elo_pre"]].rename(
    columns={
        "home_team": "team",
        "home_elo_pre": "elo"
    }
)

# Away-team Elo records
away_elo = df_match_features[["date", "away_team", "away_elo_pre"]].rename(
    columns={
        "away_team": "team",
        "away_elo_pre": "elo"
    }
)

# Combine home and away Elo records
df_team_elo = pd.concat([home_elo, away_elo], ignore_index=True)

# Get latest available Elo before the 2026 tournament
df_latest_elo = (
    df_team_elo
    .sort_values("date")
    .drop_duplicates(subset="team", keep="last")
    .reset_index(drop=True)
)

team_to_elo = dict(zip(df_latest_elo["team"], df_latest_elo["elo"]))

df_latest_elo.tail()

,date,team,elo
317,2026-03-31,Republic of Ireland,1748.177082
318,2026-03-31,Russia,1840.274201
319,2026-03-31,Spain,2229.756779
320,2026-03-31,Curaçao,1621.162116
321,2026-03-31,Comoros,1488.747627


In [12]:
#### create confederation lookup

team_to_confederation = dict(
    zip(df_confederations["nation"], df_confederations["confederation"])
)

list(team_to_confederation.items())[:5]

[('France', 'UEFA'),
 ('Spain', 'UEFA'),
 ('England', 'UEFA'),
 ('Portugal', 'UEFA'),
 ('Netherlands', 'UEFA')]

In [13]:
#### create form lookup

team_to_form = dict(
    zip(df_form["team"], df_form["form_score"])
)

list(team_to_form.items())[:5]

[('Morocco', 63.0),
 ('Japan', 61.0),
 ('Spain', 59.0),
 ('Portugal', 56.0),
 ('England', 54.0)]

In [15]:
#### create FIFA ranking lookup

team_to_fifa_rank = dict(
    zip(df_fifa_rank["Nation"], df_fifa_rank["FIFA_2026_rank"])
)

team_to_fifa_rank_change = dict(
    zip(df_fifa_rank["Nation"], df_fifa_rank["rank_change"])
)

list(team_to_fifa_rank.items())[:8]

[('France', 1),
 ('Spain', 2),
 ('Argentina', 3),
 ('England', 4),
 ('Portugal', 5),
 ('Brazil', 6),
 ('Netherlands', 7),
 ('Morocco', 8)]

In [16]:
#### create H2H helper function

def get_h2h_score(team_a, team_b):
    """
    Return the head-to-head score from team_a's perspective.

    Example:
    If Argentina vs Brazil = +6,
    then Brazil vs Argentina = -6.
    """

    direct_match = df_h2h[
        (df_h2h["team_a"] == team_a) &
        (df_h2h["team_b"] == team_b)
    ]

    if len(direct_match) > 0:
        return direct_match["h2h_score"].iloc[0]

    reverse_match = df_h2h[
        (df_h2h["team_a"] == team_b) &
        (df_h2h["team_b"] == team_a)
    ]

    if len(reverse_match) > 0:
        return -reverse_match["h2h_score"].iloc[0]

    return 0

In [18]:
#### test H2H helper

test_pairs = [
    ("Argentina", "Brazil"),
    ("Brazil", "Argentina"),
    ("France", "Spain"),
    ("Spain", "France"),
    ("France", "Senegal"),
    ("Algeria", "Morocco")
]

for team_a, team_b in test_pairs:
    print(f"{team_a} vs {team_b}: {get_h2h_score(team_a, team_b)}")

Argentina vs Brazil: 6.0
Brazil vs Argentina: -6.0
France vs Spain: -2.0
Spain vs France: 2.0
France vs Senegal: 0.0
Algeria vs Morocco: -6.0


In [19]:
#### sanity check all World Cup teams have lookup values

wc_teams = sorted(df_groups["nation"].unique())

missing_elo = [team for team in wc_teams if team not in team_to_elo]
missing_confederation = [team for team in wc_teams if team not in team_to_confederation]
missing_form = [team for team in wc_teams if team not in team_to_form]
missing_fifa_rank = [team for team in wc_teams if team not in team_to_fifa_rank]

print("Missing Elo:", missing_elo)
print("Missing confederation:", missing_confederation)
print("Missing form:", missing_form)
print("Missing FIFA rank:", missing_fifa_rank)

Missing Elo: []
Missing confederation: []
Missing form: []
Missing FIFA rank: []


In [21]:
#### inspect one team profile

sample_team = "Morocco"

team_profile = {
    "team": sample_team,
    "elo": team_to_elo.get(sample_team),
    "confederation": team_to_confederation.get(sample_team),
    "form_score": team_to_form.get(sample_team),
    "fifa_rank": team_to_fifa_rank.get(sample_team),
    "rank_change": team_to_fifa_rank_change.get(sample_team),
}

team_profile

{'team': 'Morocco',
 'elo': 1972.941440105271,
 'confederation': 'CAF',
 'form_score': 63.0,
 'fifa_rank': 8,
 'rank_change': 3}

#### Choice:
- Keep the prediction model pure. Use only the features it was trained and validated on. 
- Use FIFA rank, form, and H2H only for dashboard context and explanations.

#### Use the model for probabilities
Poisson model:
- Elo + match context → expected goals → win/draw/loss probabilities

#### Use the extra football context for interpretation:
Dashboard explanation:
- FIFA rank, rank change, recent form, H2H, confederation, Elo difference

PS: I also calculated recent form, FIFA rankings, and head-to-head records. But I did not feed them into the model because they were not part of the validated training setup. Instead, I use them in the dashboard to explain each matchup alongside the model probabilities